# Naive RAG — pełny kontekst CSV

Najprostsze możliwe podejście: bierzemy pierwsze pytanie z `golden_questions.json` i wysyłamy je do modelu razem z **całym** `products_zurada.csv` wklejonym do kontekstu.  
Brak retrieval, brak embeddingów — czysty brute-force.

In [11]:
%pip install openai pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import json
import pandas as pd
from pathlib import Path
from openai import OpenAI

# ── Konfiguracja ───────────────────────────────────────────────────────────────
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "Brak klucza")
MODEL              = "openai/gpt-5.4-mini"
BASE_URL           = "https://openrouter.ai/api/v1"

BASE_DIR       = Path(".")
QUESTIONS_FILE = BASE_DIR / "extended_golden_questions.json"
PRODUCTS_CSV   = BASE_DIR / "extended_products_zurada.csv"

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=BASE_URL,
)
print(f"Model: {MODEL}")
print(f"Klucz API: {'OK' if OPENROUTER_API_KEY != 'Brak klucza' else 'BRAK — ustaw OPENROUTER_API_KEY'}")

Model: openai/gpt-5.4-mini
Klucz API: OK


In [20]:
# ── Wczytaj pytanie i katalog produktów ────────────────────────────────────────
with open(QUESTIONS_FILE, encoding="utf-8") as f:
    questions = json.load(f)

question_obj = questions[4]
question     = question_obj["question"]

df = pd.read_csv(PRODUCTS_CSV)

print(f"Pytanie ({question_obj['id']}): {question}")
print(f"\nOczekiwane ID produktów:     {question_obj['expectedProductIds']}")
print(f"NIE-pasujące ID produktów:   {question_obj['expectedNoProductIds']}")
print(f"Trudność:                    {question_obj['difficulty']}")
print(f"\nKatalog: {len(df)} produktów, {len(df.columns)} kolumn")
print(f"Kolumny: {list(df.columns)}")

Pytanie (2): Jakim środkiem najlepiej usunąć przypalony tłuszcz, nagar i sadzę z rusztu grilla oraz wnętrza piekarnika?

Oczekiwane ID produktów:     [5, 34, 128]
NIE-pasujące ID produktów:   [1, 4, 43, 67]
Trudność:                    easy

Katalog: 262 produktów, 8 kolumn
Kolumny: ['product_id', 'product_name', 'title', 'content_encoded', 'short_description', 'right_description', 'technical_sheet', 'categories']


In [14]:
# ── Buduj kontekst — pełny CSV jako tekst ──────────────────────────────────────
catalog_text = df.to_csv(index=False)

# Szacunkowa liczba znaków
chars = len(catalog_text)
approx_tokens = chars // 4  # przybliżenie: ~4 znaki = 1 token
print(f"Rozmiar kontekstu CSV: {chars:,} znaków (~{approx_tokens:,} tokenów)")

Rozmiar kontekstu CSV: 281,876 znaków (~70,469 tokenów)


In [21]:
# ── Prompty systemowe i funkcja wywołania modelu ──────────────────────────────
SYSTEM_PROMPT = """Jesteś ekspertem ds. środków czystości firmy Zurada.
Poniżej otrzymasz pełny katalog produktów Zurada w formacie CSV.
Na jego podstawie odpowiedz na pytanie klienta.

Zasady:
- Odpowiadaj wyłącznie na podstawie danych z katalogu.
- Nie wymyślaj cech produktów spoza katalogu.
- Odpowiadaj po polsku.
- Zwróć uwagę na kategorię produktu (kolumna category) i jego przeznaczenie (kolumna usage).
- Jeśli żaden produkt nie pasuje — zwróć pustą listę product_ids.

Zwróć odpowiedź jako JSON:
{"product_ids": [{productId: integer, reason: "krótkie uzasadnienie po polsku"}, ...],"}] }
Jeśli nie znajdziesz pasujących produktów, napisz wyjaśnienie"""

SYSTEM_PROMPT_2 = """Jesteś rygorystycznym ekspertem ds. środków czystości firmy Zurada.
Poniżej otrzymasz katalog produktów Zurada. Twoim zadaniem jest precyzyjna odpowiedź na pytanie klienta poprzez wybór TYLKO tych produktów, które w 100% bezpiecznie i logicznie odpowiadają na jego potrzebę.

ZASADY KRYTYCZNE (ZERO TOLERANCJI DLA BŁĘDÓW):
1. Brak halucynacji dopasowujących: Odpowiadaj WYŁĄCZNIE na podstawie danych z katalogu. NIGDY nie wymyślaj i nie naginaj cech produktu, by zadowolić klienta (np. jeśli preparat jest "do stosowania okresowego" lub do "wieloletniego kamienia", BEZWZGLĘDNIE odrzuć go, gdy klient prosi o środek do mycia "codziennego").
2. Twarde wykluczenia (Bezpieczeństwo): Przed poleceniem produktu rygorystycznie sprawdzaj 'odradzane_powierzchnie' oraz logikę chemiczną. (Pamiętaj np. że marmur to kamień naturalny i niszczy się od kwasów). Jeśli jest najmniejsze ryzyko uszkodzenia powierzchni u klienta - odrzuć produkt.
3. Precyzja Domeny i Metody aplikacji:
   - Zwracaj uwagę na formę chemii i intencję (np. żel to nie piana, spray to nie płyn do wlania do zbiornika).
   - Ściśle oddzielaj chemię domową od samochodowej i przemysłowej (np. nie polecaj "Piany do mycia aut o zapachu winogron" gdy klient szuka domowego odświeżacza powietrza).
   - Ściśle oddzielaj mycie ręczne od maszynowego (nie polecaj chemii do zmywarek/wyparzarek do zwykłego mycia ręcznego).

Jeśli żaden produkt nie pasuje w pełni lub użycie jakiegokolwiek wiązałoby się z ryzykiem zniszczenia powierzchni — zwróć pustą listę product_ids.

Zwróć odpowiedź jako czysty JSON w poniższym formacie:
{
  "product_ids": [
    {
      "productId": 123, 
      "reason": "Krótkie uzasadnienie po polsku, udowadniające, dlaczego produkt spełnia wszystkie wymogi i jest bezpieczny."
    }
  ],
  "explanation": "Opcjonalne krótkie wyjaśnienie, np. jeśli lista jest pusta z powodu braku bezpiecznego produktu."
}
"""

SYSTEM_PROMPT_3 = """Jesteś ekspertem ds. środków czystości firmy Zurada.
Poniżej otrzymasz katalog produktów Zurada. Twoim zadaniem jest trafna odpowiedź na pytanie klienta, wybierając odpowiednie produkty z bazy.

Zasady analizy (Logika i Bezpieczeństwo):
1. Opieraj się w 100% na faktach z katalogu. Nie wymyślaj przeznaczenia produktu (np. jeśli preparat jest okresowy/do gruntownego mycia, nie polecaj go do mycia codziennego).
2. Zwracaj uwagę na wykluczenia (kolumna 'odradzane_powierzchnie'). Używaj logiki chemicznej (np. marmur to wrażliwy kamień naturalny, unikaj kwasów). Jeśli na podstawie katalogu dany środek jest wyraźnie niebezpieczny dla powierzchni klienta - pomiń go.
3. Czytaj intencję klienta: odróżniaj środki do mycia ręcznego od maszynowego (zmywarki) oraz formy aplikacji (płyn do zbiornika vs spray, żel vs piana).
4. Jeśli dany środek ogólnie pasuje do czyszczonej powierzchni (np. metal, naczynia) i nie ma wyraźnych przeciwwskazań w katalogu, możesz go polecić. 

Jeśli żaden produkt nie pasuje lub użycie każdego grozi zniszczeniem powierzchni — zwróć pustą listę product_ids.

Format odpowiedzi (Czysty JSON):
- Wygeneruj poprawny technicznie format JSON.
- W polu "reason" napisz MAKSYMALNIE JEDNO krótkie zdanie.
- W polu "reason" absolutnie NIE UŻYWAJ cudzysłowów ("), używaj apostrofów ('), aby nie zepsuć struktury JSON.

{
  "product_ids": [
    {
      "productId": 123, 
      "reason": "Krótkie uzasadnienie bez cudzyslowow."
    }
  ]
}
"""

SYSTEM_PROMPT_4 = """Jesteś ekspertem ds. środków czystości firmy Zurada.
Poniżej otrzymasz katalog produktów Zurada. Twoim zadaniem jest trafna odpowiedź na pytanie klienta, wybierając odpowiednie produkty z bazy.

Zasady analizy (Logika i Bezpieczeństwo):
1. Opieraj się w 100% na faktach z katalogu. Nie wymyślaj przeznaczenia produktu (np. jeśli preparat jest okresowy/do gruntownego mycia, nie polecaj go do mycia codziennego).
2. Zwracaj uwagę na wykluczenia (kolumna 'odradzane_powierzchnie'). Używaj logiki chemicznej (np. marmur to wrażliwy kamień naturalny, unikaj kwasów). Jeśli na podstawie katalogu dany środek jest wyraźnie niebezpieczny dla powierzchni klienta - pomiń go.
3. Czytaj intencję klienta: odróżniaj środki do mycia ręcznego od maszynowego (zmywarki) oraz formy aplikacji (płyn do zbiornika vs spray, żel vs piana).
4. Dopasuj kontekst i skalę zastosowania: Odróżniaj codzienne zastosowania domowe od specjalistycznej chemii przemysłowej, warsztatowej lub gastronomicznej (HoReCa). Jeśli klient pyta o domowy sprzęt (np. domowy grill), unikaj preparatów profilowanych pod ciężkie maszyny czy specyficzne urządzenia dla gastronomii (np. piece wędzarnicze).
5. Jeśli dany środek ogólnie pasuje do czyszczonej powierzchni (np. metal, naczynia) i nie ma wyraźnych przeciwwskazań w katalogu, możesz go polecić. 

Jeśli żaden produkt nie pasuje lub użycie każdego grozi zniszczeniem powierzchni — zwróć pustą listę product_ids.

Format odpowiedzi (Czysty JSON):
- Wygeneruj poprawny technicznie format JSON.
- W polu "reason" napisz MAKSYMALNIE JEDNO krótkie zdanie.
- W polu "reason" absolutnie NIE UŻYWAJ cudzysłowów ("), używaj apostrofów ('), aby nie zepsuć struktury JSON.

{
  "product_ids": [
    {
      "productId": 123, 
      "reason": "Krótkie uzasadnienie bez cudzyslowow."
    }
  ]
}
"""

def ask_question(question: str, system_prompt: str, json_mode: bool = False, max_tokens: int = 10000):
    """Wysyła pytanie z pełnym katalogiem CSV do modelu. Zwraca (odpowiedź, usage)."""
    user_msg = f"Katalog produktów Zurada (CSV):\n\n{catalog_text}\n\n---\n\nPytanie klienta: {question}"
    kwargs = dict(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_msg},
        ],
        max_completion_tokens=max_tokens,
    )
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content or "", resp.usage

print("ask_question gotowa.")

ask_question gotowa.


In [22]:
print("Wysyłam zapytanie do modelu...")
raw, usage = ask_question(question, SYSTEM_PROMPT_4, json_mode=True)
data = json.loads(raw)  # .format() usuwa ewentualne znaki nowej linii
returned_ids = [x["productId"] if isinstance(x, dict) else int(x) for x in data.get("product_ids", [])]

print("=" * 60)
print(f"PYTANIE: {question}")
print("=" * 60)
print()
print("ODPOWIEDŹ MODELU:")
print(data)
print(f"Zwrócone product_id: {returned_ids}")
print()
print("-" * 60)
print(f"Oczekiwane product_id:   {question_obj['expectedProductIds']}")
print(f"NIE-pasujące product_id: {question_obj['expectedNoProductIds']}")
print()
print("UZASADNIENIE (oczekiwane):")
for exp in question_obj["explanation"]:
      print(f"  [{exp['productId']}] {exp['reasoning']}")
print()
print(f"Tokeny: prompt={usage.prompt_tokens}, odpowiedź={usage.completion_tokens}, razem={usage.total_tokens}")

Wysyłam zapytanie do modelu...
PYTANIE: Jakim środkiem najlepiej usunąć przypalony tłuszcz, nagar i sadzę z rusztu grilla oraz wnętrza piekarnika?

ODPOWIEDŹ MODELU:
{'product_ids': [{'productId': 5, 'reason': 'Jest przeznaczony do grillów, piekarników, rusztów i usuwa przypalenia bez szorowania.'}, {'productId': 34, 'reason': 'Nadaje się do piekarników, grilli i usuwa przypalenia, tłuszcz oraz sadzę.'}, {'productId': 128, 'reason': 'Usuwa tłuszcz, nagar i sadzę z piekarników oraz grilli.'}]}
Zwrócone product_id: [5, 34, 128]

------------------------------------------------------------
Oczekiwane product_id:   [5, 34, 128]
NIE-pasujące product_id: [1, 4, 43, 67]

UZASADNIENIE (oczekiwane):
  [5] Pasuje 'Zurada Piana Moc', ponieważ jest to preparat dedykowany dokładnie do tego celu: mycia grillów, piekarników, rusztów i radzi sobie z przypaleniami.
  [34] Pasuje 'Zurada ŻarMoc', ponieważ to ekologiczna alternatywa (pianka) stworzona do usuwania ciężkich przypaleń i tłuszczu właśnie z p

In [23]:
# ── Pętla przez wszystkie pytania — expected vs returned product_id ────────────
import time

SLEEP_BETWEEN = 0.1

results = []
total_tokens = 0

for q in questions:
    qid        = q["id"]
    expected   = set(q["expectedProductIds"])
    no_product = len(expected) == 0

    try:
        raw, usage = ask_question(q["question"], SYSTEM_PROMPT_4, json_mode=True, max_tokens=400)
        data         = json.loads(raw)
        returned_ids = [x["productId"] if isinstance(x, dict) else int(x) for x in data.get("product_ids", [])]
        ans_text     = data.get("answer", "")
        returned     = set(returned_ids)
        total_tokens += usage.total_tokens
    except Exception as e:
        print(f"  [BŁĄD] pytanie {qid}: {e}")
        returned, ans_text = set(), str(e)

    hit     = expected & returned
    missed  = expected - returned
    extra   = returned - expected
    correct = returned == expected

    results.append({
        "id":         qid,
        "difficulty": q.get("difficulty", ""),
        "no_product": no_product,
        "expected":   sorted(expected),
        "returned":   sorted(returned),
        "hit":        sorted(hit),
        "missed":     sorted(missed),
        "extra":      sorted(extra),
        "correct":    correct,
        "answer":     ans_text[:120],
    })

    tag    = "[∅]" if no_product else "   "
    status = "✓" if correct else "✗"
    print(f"[{status}]{tag} Q{qid:02d} oczekiwane={sorted(expected) or '∅'} zwrócone={sorted(returned) or '∅'}")
    time.sleep(SLEEP_BETWEEN)

print(f"\nŁączne tokeny: {total_tokens:,}")

[✗]    Q01 oczekiwane=[12, 82, 139] zwrócone=[82, 139, 140]
[✗]    Q08 oczekiwane=[136, 137, 189, 190] zwrócone=[137, 189, 190]
[✓]    Q09 oczekiwane=[233, 257] zwrócone=[233, 257]
[✗]    Q10 oczekiwane=[213, 217, 218] zwrócone=[213, 218]
[✗]    Q02 oczekiwane=[5, 34, 128] zwrócone=[5, 34]
[✓]    Q03 oczekiwane=[45, 112, 152] zwrócone=[45, 112, 152]
[✗]    Q04 oczekiwane=[11] zwrócone=[11, 12, 25, 29, 80, 82, 133, 139]
[✗]    Q05 oczekiwane=[2, 39, 41, 129] zwrócone=[41, 129, 164]
[✓]    Q06 oczekiwane=[7, 69] zwrócone=[7, 69]

Łączne tokeny: 828,517


In [24]:
# ── Wielokrotne powtórzenia każdego pytania (do analizy stabilności) ───────────
import time
from datetime import datetime

N_RUNS        = 8
SLEEP_BETWEEN = 0.1
OUTPUT_FILE   = BASE_DIR / "repeated_runs.json"

repeated_results = []
total_tokens_repeated = 0

for q in questions:
    qid = q["id"]
    runs = []

    for run_idx in range(1, N_RUNS + 1):
        try:
            raw, usage = ask_question(q["question"], SYSTEM_PROMPT_4, json_mode=True, max_tokens=400)
            data         = json.loads(raw)
            returned_ids = [x["productId"] if isinstance(x, dict) else int(x) for x in data.get("product_ids", [])]
            ans_text     = data.get("answer", "")
            total_tokens_repeated += usage.total_tokens
            runs.append({
                "run":               run_idx,
                "returned_ids":      returned_ids,
                "answer":            ans_text,
                "model":             MODEL,
                "prompt_tokens":     usage.prompt_tokens,
                "completion_tokens": usage.completion_tokens,
                "total_tokens":      usage.total_tokens,
            })
        except Exception as e:
            runs.append({"run": run_idx, "error": str(e)})

        time.sleep(SLEEP_BETWEEN)

    repeated_results.append({
        "id":                 qid,
        "question":           q["question"],
        "expectedProductIds": q["expectedProductIds"],
        "expectedNoProductIds": q["expectedNoProductIds"],
        "difficulty":         q.get("difficulty", ""),
        "runs":               runs,
    })

    all_returned = [set(r["returned_ids"]) for r in runs if "returned_ids" in r]
    expected_set = set(q["expectedProductIds"])
    n_correct    = sum(1 for r in all_returned if r == expected_set)
    print(f"Q{qid:02d} [{q['difficulty']:4s}] {n_correct}/{N_RUNS} poprawnych — oczekiwane={sorted(expected_set)}")

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "model":     MODEL,
        "n_runs":    N_RUNS,
        "questions": repeated_results,
    }, f, ensure_ascii=False, indent=2)

print(f"\nZapisano: {OUTPUT_FILE}")
print(f"Łączne tokeny: {total_tokens_repeated:,}")

Q01 [easy] 5/8 poprawnych — oczekiwane=[12, 82, 139]
Q08 [easy] 1/8 poprawnych — oczekiwane=[136, 137, 189, 190]
Q09 [easy] 1/8 poprawnych — oczekiwane=[233, 257]
Q10 [easy] 7/8 poprawnych — oczekiwane=[213, 217, 218]
Q02 [easy] 4/8 poprawnych — oczekiwane=[5, 34, 128]
Q03 [hard] 5/8 poprawnych — oczekiwane=[45, 112, 152]
Q04 [hard] 1/8 poprawnych — oczekiwane=[11]
Q05 [easy] 0/8 poprawnych — oczekiwane=[2, 39, 41, 129]
Q06 [easy] 7/8 poprawnych — oczekiwane=[7, 69]

Zapisano: repeated_runs.json
Łączne tokeny: 6,627,368


In [25]:
# ── Podsumowanie wyników ───────────────────────────────────────────────────────
results_df = pd.DataFrame(results)

n_total      = len(results_df)
n_correct    = results_df["correct"].sum()
accuracy     = n_correct / n_total * 100

normal_df    = results_df[~results_df["no_product"]]
noproduct_df = results_df[results_df["no_product"]]

print("=" * 60)
print(f"WYNIK OGÓLNY: {n_correct}/{n_total} poprawnych ({accuracy:.1f}%)")
print(f"  Pytania z produktem:      {normal_df['correct'].sum()}/{len(normal_df)}")
print(f"  Pytania 'brak produktu':  {noproduct_df['correct'].sum()}/{len(noproduct_df)}")
print("=" * 60)
print()

display_cols = ["id", "no_product", "expected", "returned", "correct", "missed", "extra"]
print(results_df[display_cols].to_string(index=False))
print()

errors = results_df[~results_df["correct"]]
if not errors.empty:
    print(f"BŁĘDY ({len(errors)} pytań):")
    for _, row in errors.iterrows():
        tag = " [∅ model powinien odmówić]" if row["no_product"] else ""
        print(f"  Q{row['id']:02d}{tag}")
        print(f"    oczekiwane: {row['expected'] or '∅'} | zwrócone: {row['returned'] or '∅'}")
        print(f"    odpowiedź: {row['answer']}")


WYNIK OGÓLNY: 3/9 poprawnych (33.3%)
  Pytania z produktem:      3/9
  Pytania 'brak produktu':  0/0

 id  no_product             expected                           returned  correct  missed                          extra
  1       False        [12, 82, 139]                     [82, 139, 140]    False    [12]                          [140]
  8       False [136, 137, 189, 190]                    [137, 189, 190]    False   [136]                             []
  9       False           [233, 257]                         [233, 257]     True      []                             []
 10       False      [213, 217, 218]                         [213, 218]    False   [217]                             []
  2       False         [5, 34, 128]                            [5, 34]    False   [128]                             []
  3       False       [45, 112, 152]                     [45, 112, 152]     True      []                             []
  4       False                 [11] [11, 12, 25, 29, 80, 